# EDA RAG Document Stats

Statistical overview for RAG document collections.

Steps:
- Inventory document directories.
- Compute file size and length summaries.
- Preview a sample document.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

rag_dirs = [
    REPO_ROOT / 'data' / 'docs',
    REPO_ROOT / 'data' / 'samples' / 'docs',
]

summary = {
    'directories': [],
    'samples': [],
    'sizes': {},
}

for rag_dir in rag_dirs:
    print('RAG dir:', rag_dir)
    if not rag_dir.exists():
        print('Missing:', rag_dir)
        continue
    files = [p for p in rag_dir.rglob('*') if p.is_file()]
    summary['directories'].append({'path': str(rag_dir), 'file_count': len(files)})
    sizes = [p.stat().st_size for p in files]
    summary['sizes'][str(rag_dir)] = {
        'file_count': len(files),
        'total_bytes': sum(sizes),
        'avg_bytes': round(sum(sizes) / len(sizes), 2) if sizes else 0,
        'max_bytes': max(sizes) if sizes else 0,
    }
    for sample in files[:5]:
        summary['samples'].append(str(sample.relative_to(REPO_ROOT)))
    print('Files:', len(files))


In [ ]:
# Preview a sample doc if available.
preview_path = None
for rel_path in summary['samples']:
    path = REPO_ROOT / rel_path
    if path.suffix.lower() in {'.md', '.txt'}:
        preview_path = path
        break

if preview_path and preview_path.exists():
    print('Preview:', preview_path.relative_to(REPO_ROOT))
    print(preview_path.read_text(encoding='utf-8', errors='ignore')[:2000])
else:
    print('No text sample found for preview')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_rag_doc_stats_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize rag-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'rag' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No rag entries found in TRAINING_DATA.json')
    else:
        print('rag datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
